# Section 3 – Unsupervised learning – clustering
In this unsupervised learning task, the objective is to cluster Reddit comments based on the words used
within the comments in them. The aim is to identify patterns and group comments with similar words
together.

In [ ]:
%pip install nltk
import nltk
nltk.download('punkt')
nltk.download('wordnet')
%pip install WordCloud

In [ ]:
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score, adjusted_rand_score, normalized_mutual_info_score, adjusted_mutual_info_score

from wordcloud import WordCloud

In [ ]:
data = pd.read_csv(f"without_stopwords_150.csv")

#we actually forgot to remove duplicates/empty rows after removing stopwords, so "we do it now" by creating a mask
mask=[]
for index, row in data.iterrows():
    try:
        if len(row['body'].strip()) == 0:
            mask.append(False)
        else:            
            mask.append(True)
    except:#if NaN
        mask.append(False)
data = data[mask]

#read npy file (pre-processing was done in 'unsupervised-pre-processing.ipynb')
X = np.load('X_reduced_150_lemmatizer_BoW.npy')
X = X[mask]

## KMeans
### Determine the number of clusters

In [ ]:
clustering_results = []
k_values=pd.concat([pd.Series(range(3, 10, 1))])

#Performing KMeans for different values of k
for k in k_values:
    kmeans = KMeans(
    init='random', #centroids are initialized randomly
    random_state=42, 
    n_init='auto', #number of times the k-means algorithm will be run with different centroid seeds. 'auto' uses a default value based on init
    max_iter=300,
    tol=1e-2,
    n_clusters=k
    )
    clustering_results.append(kmeans.fit(X))
    #print(f"Fitted KMeans with {k} clusters")#just to have a reference to how much time it's taking to execute

#Computing silhouette scores
n_sample=10000#number of samples to use for silhouette calculation, to speed up the process
#n_sample = 5 
# Plotting the results
fig, axs = plt.subplots(1,2, figsize=(8, 3.5))
axs[0].set_xlabel("Number of clusters k")
axs[0].set_ylabel("Clustering error")
axs[0].grid()
plt.title("KMeans clustering evaluation")
axs[1].set_xlabel("Number of clusters k")
axs[1].set_ylabel("Silhouette")
axs[1].grid()
plt.tight_layout()
for i in range(3):#execute more than once to understand how randomness (of silhouette sampling) affects the results
    silhouetteScores = []
    print(f"Iteration {i+1} of silhouette score calculation")
    for result in clustering_results:
        try:
            silhouetteScores.append(silhouette_score(X, result.labels_, sample_size=n_sample))
            print(f"silhouette score computed k={len(result.labels_)}: {silhouetteScores[-1]}")
        except ValueError as e:
            print(f"Skipping silhouette calculation due to error: {e}")
            silhouetteScores.append(-1)
    if i==0:#plot the error only the first time, it's the same for all iterations
        axs[0].plot(
            [k for k in k_values], 
            [result.inertia_ for result in clustering_results]
        )
    axs[1].plot(
        [k for k in k_values], 
        [s for s in silhouetteScores]
    )
plt.show()

In [ ]:
#Plot the number of comments for each cluster
for i in clustering_results:
    cluster_sizes = pd.Series(i.labels_).value_counts()
    plt.figure(figsize=(10, 5))
    plt.bar(cluster_sizes.index, cluster_sizes.values)
    plt.title(f"Number of Comments in Each Cluster (k={len(cluster_sizes)})")
    plt.xlabel("Cluster")
    plt.ylabel("Number of Comments")
    plt.xticks(cluster_sizes.index)
    plt.bar_label(plt.bar(cluster_sizes.index, cluster_sizes.values))
    plt.show()
    #print("Cluster sizes:\n", cluster_sizes)

In [ ]:
numberOfChoosenClusters=3
indexOfChoosenCluster=k_values[k_values==numberOfChoosenClusters].index[0]
data["cluster"]= clustering_results[indexOfChoosenCluster].labels_

### User Distribution Analysis
Analyze how the users are distributed (each user can be assigned to more clusters because we
are clustering comments). 
#### Show the ECDF of number of clusters assigned to each user.

In [ ]:
user_clusters = data.groupby('author')['cluster'].nunique().reset_index()#grouping by author
#user clusters is now a series with author and number of clusters
user_clusters.columns = ['author', 'num_clusters']#turn it into a DF

# Filter user_clusters to only those authors who have multiple comments
# authors_with_multiple_comments = data.groupby('author')['body'].nunique()
# authors_with_multiple_comments = authors_with_multiple_comments[authors_with_multiple_comments > 1].index
# user_clusters = user_clusters[user_clusters['author'].isin(authors_with_multiple_comments)]

user_comments = data.groupby('author')['body'].nunique().reset_index()
user_comments.columns = ['author', 'num_comments']

plt.figure(figsize=(8, 5))

sns.ecdfplot(data=user_clusters, x='num_clusters')
plt.ylabel('ECDF')
plt.xlabel('Clusters')
plt.title('ECDF of number of clusters per user and number of comments per user')

#plot ECDF of number of comments per user
sns.ecdfplot(data=user_comments, x='num_comments')
plt.grid(True)
x_min = 1
x_max = user_clusters['num_clusters'].max()+1
plt.xlim(0.8, x_max+0.2)
plt.xticks(range(0, x_max))
plt.legend(['Number of Clusters per User', 'Number of Comments per User'])

plt.show()

#### Use the first two components of the ~~PCA~~ TruncatedSVD. Then visualize on these two dimensions to gain insights into their distribution and separability

In [ ]:
unique_clusters = np.sort(data['cluster'].unique())
cluster_to_idx = {cluster: idx for idx, cluster in enumerate(unique_clusters)}
color_idx = data['cluster'].map(cluster_to_idx)

cmap = plt.get_cmap('tab10', len(unique_clusters))
norm = plt.matplotlib.colors.BoundaryNorm(
    boundaries=np.arange(len(unique_clusters) + 1) - 0.5,
    ncolors=len(unique_clusters)

)

plt.figure(figsize=(8, 6))
scatter = plt.scatter(X[:, 0], X[:, 1], c=color_idx, cmap=cmap, norm=norm, alpha=0.6)
plt.xlabel('Truncated SVD Component 1')
plt.ylabel('Truncated SVD Component 2')
plt.title('SVD: First Two Components Colored by Cluster')
cbar = plt.colorbar(scatter, ticks=np.arange(len(unique_clusters)))
cbar.set_ticklabels(unique_clusters)
cbar.set_label('Cluster', rotation=270, labelpad=15)
plt.grid(True)
plt.show()

### Cluster analysis
Analyze the characteristics of each cluster. This might involve examining the most frequent
words in each cluster (try word cloud). HINT: the most used words are not always the most
representative, for example word ‘the’ will be very popular in every cluster if you didn’t remove it from the dataset. Try to understand which are the most representative

In [ ]:
#Group by cluster and concatenate all comments in each cluster, then compute TF-IDF to understand important words in each cluster
cluster_documents=data[['body','cluster']].groupby('cluster').agg(' '.join).reset_index()
tfidf_vectorizer = TfidfVectorizer()
tfidf_matrix = tfidf_vectorizer.fit_transform(cluster_documents['body'])
terms = tfidf_vectorizer.get_feature_names_out()

#For each cluster, get the top 50 words by TF-IDF score and plot a word cloud
for cluster_num in range(numberOfChoosenClusters):
    tfidf_scores = tfidf_matrix[cluster_num].toarray().flatten()
    top_indices = np.argsort(tfidf_scores)[-50:]  # Get indices of top 50 words
    top_terms = {terms[i]: tfidf_scores[i] for i in top_indices}

    wordcloud = WordCloud(width=800, height=400, background_color='white').generate_from_frequencies(top_terms)

    plt.figure(figsize=(10, 5))
    plt.imshow(wordcloud, interpolation='bilinear')
    plt.axis('off')
    plt.title(f'Word Cloud for Cluster {cluster_num}')
    plt.show()

Finally, analyze and visualize how the subreddit are distributed among the found clusters. Are there pure clusters where only few related
subreddits end up?

In [ ]:
subreddit_cluster = data.groupby('subreddit')['cluster'].nunique().reset_index()#grouping by subreddit
#subreddit clusters is now a series with subreddit and number of clusters
subreddit_cluster.columns = ['subreddit', 'num_clusters']#turn it into a DF

# Filter subreddit_cluster to only those subreddits that have multiple comments
subreddits_with_multiple_comments = data.groupby('subreddit')['body'].nunique()
subreddits_with_multiple_comments = subreddits_with_multiple_comments[subreddits_with_multiple_comments > 1].index
subreddit_cluster = subreddit_cluster[subreddit_cluster['subreddit'].isin(subreddits_with_multiple_comments)]

plt.figure(figsize=(8, 5))
sns.ecdfplot(data=subreddit_cluster, x='num_clusters')
plt.title('ECDF of Number of Clusters per Subreddit')
plt.xlabel('Number of Clusters Assigned to Subreddit')
plt.ylabel('ECDF')
plt.xticks(range(1, subreddit_cluster['num_clusters'].max()))
plt.grid(True)
plt.show()

In [ ]:
subredditsNames = []

#compute TF-IDF for subreddits in each cluster, to understand which subreddits are more important in each cluster,
#then plot a word cloud for each cluster
for i in range(0, numberOfChoosenClusters):
    cluster_sizes = data['cluster'].value_counts()
    print(f"Cluster {i} has {cluster_sizes[i]} comments")
    subredditsNames.append(" ".join(data[data['cluster'] == i]['subreddit'].values))


vectorizer = TfidfVectorizer(max_df=0.95)#max_df=0.95 means ignore subreddits that appear in 95% of clusters (since we have only 3 clusters, it means ignore subreddits that appear in all clusters)
subreddit_TFIDF = vectorizer.fit_transform(subredditsNames)


for i in range(0, numberOfChoosenClusters):

    doc_weights = np.asarray(subreddit_TFIDF[i].toarray()).ravel()
    freq_doc = {t: w for t, w in zip(vectorizer.get_feature_names_out(), doc_weights) if w > 0}
    wordcloud = WordCloud(width=800, height=400, background_color="white").generate_from_frequencies(freq_doc)

    #wordcloud = WordCloud(width=800, height=400, background_color='white').generate(subreddit_TFIDF.toarray()[i])

    plt.figure(figsize=(10, 5))
    plt.imshow(wordcloud, interpolation='bilinear')
    plt.axis('off')  
    plt.title(f"Cluster {i} subreddits word cloud")
    plt.show()

#### Do clusters reflect subreddit division?

In [ ]:
ari = adjusted_rand_score(data["subreddit"], data["cluster"])
nmi = normalized_mutual_info_score(data["subreddit"], data["cluster"])
ami = adjusted_mutual_info_score(data["subreddit"], data["cluster"])

print("ARI:", ari)
print("NMI:", nmi)
print("AMI:", ami)

Another thing we can try is to analyze how long are comments in the different clusters, the idea is that the shorter comments are closer to "0" than the longer cluster, considering the trend in length of comments the biggest cluster may contain those shorter comments

In [ ]:
#plot for each cluster the length of comments (boxplot)
plt.figure(figsize=(10, 6))
for i in range(numberOfChoosenClusters):
    cluster_data = data[data['cluster'] == i]
    sns.boxplot(x=cluster_data['cluster'], y=cluster_data['body'].apply(lambda x: len(x.split())))
plt.xlabel('Cluster')

#print for each cluster average length of comments, max, min 25th percentile, 75th percentile
for i in range(numberOfChoosenClusters):
    cluster_data = data[data['cluster'] == i]
    comment_lengths = cluster_data['body'].apply(lambda x: len(x.split()))
    print(f"Cluster {i}:")
    print(f"  Min length: {comment_lengths.min()} words")
    print(f"  25th percentile: {comment_lengths.quantile(0.25)} words")
    print(f"  Average length: {comment_lengths.mean():.2f} words")
    print(f"  75th percentile: {comment_lengths.quantile(0.75)} words")
    print(f"  Max length: {comment_lengths.max()} words")


In [ ]:
#store cluster labels in case we want to analyze further later
np.save('kmeans_cluster_labels.npy', data['cluster'].values)